[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srini2k03/jobpost-extractor-qlora/blob/main/notebooks/01_paraphrase_7b.ipynb)

# 01 · Paraphrase pass (7B) + real test set drafts

Rewrites a subset of synthetic training postings with **Qwen2.5-7B-Instruct (4-bit)** for realism, keeps only rewrites that pass the rule + NLI checker, then fetches ~200 real public postings and drafts their labels with the same 7B model (few-shot) for human verification with `jobx verify`. Same code as `make local-paraphrase` (`jobx.pipeline.paraphrase`); re-running continues where it stopped.

**How to run**
1. Runtime → Change runtime type → **T4 GPU**.
2. Secrets (key icon in the left sidebar), each with *Notebook access* switched on: `HF_TOKEN`, `HF_USERNAME`.
   `HF_USERNAME` is your Hugging Face account name; `HF_TOKEN` needs *write* access.
3. Runtime → **Run all**. Expected runtime: **~1.5 h (paraphrase ~60 min for 600 postings, drafts ~20 min)**. Cells after the setup are safe to re-run.

**Outputs:** `data/paraphrased/paraphrased_train.jsonl` + `filter_stats.json`, `data/real/test_real_draft.jsonl` (+ `raw_meta.json`), all pushed to the private dataset repo `<HF_USERNAME>/jobx-artifacts`.

All code lives in `src/jobx` of the GitHub repo; this notebook only orchestrates it.

In [ ]:
# Setup: clone the repo (code + data), install pinned dependencies, install jobx itself.
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/srini2k03/jobpost-extractor-qlora.git"
if IN_COLAB:
    if not os.path.exists("/content/jobpost-extractor-qlora"):
        !git clone --depth 1 {REPO_URL} /content/jobpost-extractor-qlora
    %cd /content/jobpost-extractor-qlora
    !pip install -q transformers==4.56.2 peft==0.17.1 trl==0.23.1 bitsandbytes==0.48.1 accelerate==1.10.1 datasets==4.1.1 huggingface_hub==0.35.3 wandb==0.22.2 lm-format-enforcer==0.11.3
    !pip install -q -e .
else:
    # running locally from a checkout: notebooks/ -> repo root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Secrets -> environment variables (read by jobx.hub and wandb).
import os
for key in ("HF_TOKEN", "HF_USERNAME", "WANDB_API_KEY"):
    if not os.environ.get(key):
        try:
            from google.colab import userdata
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"secret {key} not available")
from jobx.hub import hf_username, repo_id
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Hub account:", hf_username(), "| artifacts repo:", repo_id("artifacts"))

## Settings
Sizes come from `configs/gpu/colab_t4.yaml` (600 rewrites, batch 8).

In [ ]:
import json
from jobx.pipeline import load_profile, paraphrase
PROFILE = load_profile('colab_t4')
DO_REAL_DRAFTS = True   # fetch ~200 real postings and draft their labels
print(PROFILE.paraphrase)

## Paraphrase, validate (rules + NLI), draft real labels, push
The real-postings fetch picks the first candidate in `configs/real_data.yaml` whose declared licence is in the allowlist. **A human still has to confirm the licence** (see `docs/real_dataset.md`). Drafts go to a *private* dataset repo; verify them with `jobx verify`.

In [ ]:
stats = paraphrase(PROFILE, push=True, do_real_drafts=DO_REAL_DRAFTS)
print(json.dumps(stats, indent=2))

## Next
On your machine: `jobx pull` (downloads these files), `jobx verify` (≈30 min for a sample), then `jobx build-data` to merge the paraphrases and refresh `DATA_CARD.md`; commit and push. Then run `02_baselines` and `03_train_qlora`.